## 0 · Environment Setup

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path so `src` is importable.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Colab-only: download dataset via kagglehub ──────────────────────────────
import os
IN_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()

if IN_COLAB:
    import kagglehub
    path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
    os.environ["DATA_DIR"] = str(Path(path) / "chest_xray")
    print(f"Dataset downloaded to: {os.environ['DATA_DIR']}")

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision import datasets

from src.config import CLASS_NAMES, DATA_DIR, DEVICE, IN_COLAB, RESULTS_DIR
from src.datasets import compute_class_weights, get_dataloaders, get_transforms

print(f"Environment : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Data root   : {DATA_DIR}")
print(f"Device      : {DEVICE}")
print(f"Data exists : {DATA_DIR.exists()}")

## 1 · Dataset Directory Layout

In [ ]:
splits = ["train", "val", "test"]

print(f"{'Split':<8}  {'Class':<12}  {'Count':>6}")
print("-" * 32)

split_counts: dict[str, dict[str, int]] = {}
for split in splits:
    split_path = DATA_DIR / split
    if not split_path.exists():
        print(f"WARNING: {split_path} not found — skipping")
        continue
    split_counts[split] = {}
    for cls in CLASS_NAMES:
        cls_path = split_path / cls
        count = len(list(cls_path.glob("*.jpeg")) + list(cls_path.glob("*.jpg")))
        split_counts[split][cls] = count
        print(f"{split:<8}  {cls:<12}  {count:>6}")

total = sum(c for s in split_counts.values() for c in s.values())
print("-" * 32)
print(f"{'Total':<22}  {total:>6}")

## 2 · Class Distribution (Bar Chart)

In [ ]:
fig, axes = plt.subplots(1, len(split_counts), figsize=(5 * len(split_counts), 4), sharey=False)
if len(split_counts) == 1:
    axes = [axes]

for ax, (split, counts) in zip(axes, split_counts.items()):
    bars = ax.bar(CLASS_NAMES, [counts[c] for c in CLASS_NAMES], color=["steelblue", "tomato"])
    for bar, val in zip(bars, [counts[c] for c in CLASS_NAMES]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 5,
            str(val),
            ha="center",
            va="bottom",
            fontsize=11,
        )
    split_total = sum(counts.values())
    ax.set_title(f"{split.capitalize()} (n={split_total})")
    ax.set_ylabel("Image count")

fig.suptitle("Class Distribution per Split", fontsize=14, y=1.02)
fig.tight_layout()
plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3 · Sample Image Grid

In [ ]:
from PIL import Image

def show_sample_grid(split: str, n_per_class: int = 4) -> None:
    """Display n_per_class raw images for each class in a split."""
    fig, axes = plt.subplots(
        len(CLASS_NAMES), n_per_class,
        figsize=(3 * n_per_class, 3 * len(CLASS_NAMES)),
    )
    for row, cls in enumerate(CLASS_NAMES):
        cls_path = DATA_DIR / split / cls
        images = sorted(cls_path.glob("*.jpeg"))[:n_per_class]
        if not images:
            images = sorted(cls_path.glob("*.jpg"))[:n_per_class]
        for col, img_path in enumerate(images):
            ax = axes[row][col]
            ax.imshow(Image.open(img_path), cmap="gray")
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(cls, fontsize=12)
    fig.suptitle(f"Sample images — {split} split", fontsize=14)
    fig.tight_layout()
    plt.show()

show_sample_grid("train", n_per_class=4)

## 4 · Image Size & Channel Audit

In [ ]:
from PIL import Image as PILImage

sizes: list[tuple[int, int]] = []
modes: Counter = Counter()

sample_paths = list((DATA_DIR / "train" / "PNEUMONIA").glob("*.jpeg"))[:200]
sample_paths += list((DATA_DIR / "train" / "NORMAL").glob("*.jpeg"))[:200]

for p in sample_paths:
    img = PILImage.open(p)
    sizes.append(img.size)  # (W, H)
    modes[img.mode] += 1

widths  = [s[0] for s in sizes]
heights = [s[1] for s in sizes]

print(f"Sample size : {len(sizes)} images")
print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
print(f"Channel modes: {dict(modes)}")
print()
print("Note: 'L' = grayscale, 'RGB' = 3-channel.")
print("All images are loaded as RGB by ImageFolder (Torchvision converts automatically).")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.hist(widths,  bins=30, color="steelblue", edgecolor="white")
ax1.set_title("Image Width Distribution")
ax1.set_xlabel("pixels")
ax2.hist(heights, bins=30, color="tomato", edgecolor="white")
ax2.set_title("Image Height Distribution")
ax2.set_xlabel("pixels")
fig.tight_layout()
plt.show()

## 5 · DataLoader Pipeline Validation

In [ ]:
loaders = get_dataloaders(DATA_DIR)

print("DataLoaders created:")
for split, loader in loaders.items():
    n_batches = len(loader)
    n_images  = len(loader.dataset)
    print(f"  {split:<6}  {n_images:>5} images  {n_batches:>4} batches")

In [ ]:
# Load one batch from training and verify shapes and label distribution.
images_batch, labels_batch = next(iter(loaders["train"]))

print(f"Batch image tensor shape : {tuple(images_batch.shape)}")
print(f"  expected               : (batch_size, 3, 224, 224)")
print(f"Batch labels shape       : {tuple(labels_batch.shape)}")
print(f"Label values in batch    : {sorted(labels_batch.unique().tolist())}  ({CLASS_NAMES})")
print(f"Pixel value range        : [{images_batch.min():.3f}, {images_batch.max():.3f}]  (normalized)")
print()
label_counts = Counter(labels_batch.tolist())
print("Label distribution in this batch:")
for idx, name in enumerate(CLASS_NAMES):
    print(f"  {name:<12} idx={idx}  count={label_counts.get(idx, 0)}")

In [ ]:
# Display a 4×4 grid of normalized batch images (un-normalized for viewing).
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize(t: torch.Tensor) -> np.ndarray:
    """Reverse ImageNet normalization for display."""
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

n_show = min(16, len(images_batch))
ncols  = 4
nrows  = (n_show + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
axes = axes.flatten()

for i in range(n_show):
    axes[i].imshow(denormalize(images_batch[i]), cmap="gray")
    axes[i].set_title(CLASS_NAMES[labels_batch[i].item()], fontsize=9)
    axes[i].axis("off")

for i in range(n_show, len(axes)):
    axes[i].set_visible(False)

fig.suptitle("Augmented training batch (un-normalized for display)", fontsize=13)
fig.tight_layout()
plt.show()

## 6 · Class Weights for CrossEntropyLoss

In [ ]:
from torchvision import datasets as tvdatasets

train_ds = tvdatasets.ImageFolder(
    root=str(DATA_DIR / "train"),
    transform=get_transforms("train"),
)

class_weights = compute_class_weights(train_ds)

print("Inverse-frequency class weights (for CrossEntropyLoss):")
for cls, w in zip(CLASS_NAMES, class_weights.tolist()):
    print(f"  {cls:<12}  weight = {w:.4f}")

print()
print("Usage in training:")
print("  criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))")
print()
ratio = class_weights[0].item() / class_weights[1].item()
print(f"Normal/Pneumonia weight ratio: {ratio:.2f}x")
print("Misclassifying NORMAL is penalised more heavily to compensate for class imbalance.")

## 7 · Summary

In [ ]:
print("=" * 50)
print(" EDA SUMMARY")
print("=" * 50)
for split, counts in split_counts.items():
    total_split = sum(counts.values())
    pct_pneumonia = counts.get("PNEUMONIA", 0) / total_split * 100
    print(f"{split:<6}  total={total_split:>5}  pneumonia={pct_pneumonia:.1f}%")
print()
print(f"Batch tensor shape  : {tuple(images_batch.shape)}  ✓")
print(f"Pipeline functional : True  ✓")
print(f"Class weights ready : {class_weights.tolist()}  ✓")
print()
print("Ready to proceed to 02_train.ipynb")